# Comparing LangGraph, CrewAI and Strands Agents

Reads the six results files the tutorials' `run_eval.py` scripts wrote (`results/<agent>/<framework>.json`),
counts the agent code in each build, and re-runs the one experiment behind the biggest gap. The written
verdict is `COMPARISON.md` at the repo root; this notebook is its evidence.

It needs only the Python standard library, so it runs in any of the three tutorial environments (and the
replay at the end needs Ollama with `qwen3-agents`).

In [1]:
import json
from pathlib import Path

ROOT = Path.cwd().resolve().parent  # this notebook lives in comparison/
FRAMEWORKS = ("langgraph", "crewai", "strands")
results = {(agent, fw): json.loads((ROOT / "results" / agent / f"{fw}.json").read_text())
           for agent in ("shipfast", "research") for fw in FRAMEWORKS}
for (agent, fw), data in results.items():
    print(f"{agent:9} {fw:10} {data['framework_version']:9} {data['model']:13} run at {data['run_at']}")

shipfast  langgraph  1.2.19    qwen3-agents  run at 2026-11-03T08:02:55+00:00
shipfast  crewai     1.15.34   qwen3-agents  run at 2026-11-03T08:34:31+00:00
shipfast  strands    1.63.0    qwen3-agents  run at 2026-11-03T09:07:16+00:00
research  langgraph  1.2.19    qwen3-agents  run at 2026-11-03T08:27:30+00:00
research  crewai     1.15.34   qwen3-agents  run at 2026-11-03T08:55:05+00:00
research  strands    1.63.0    qwen3-agents  run at 2026-11-03T09:26:39+00:00


## Summary

In [2]:
print(f"{'agent':9} {'framework':10} {'passed':>7} {'tokens':>8} {'seconds':>8} {'llm calls':>10} {'tool calls':>11}")
for (agent, fw), data in results.items():
    items = data["scenarios" if agent == "shipfast" else "questions"]
    s = data["summary"]
    print(f"{agent:9} {fw:10} {s['passed']:>3}/{s['total']:<3} {s['total_tokens']:8} {s['total_latency_s']:8.0f} "
          f"{sum(i['llm_calls'] for i in items):10} {sum(len(i['tool_calls']) for i in items):11}")

agent     framework   passed   tokens  seconds  llm calls  tool calls
shipfast  langgraph    7/8      29962      387         44          20
shipfast  crewai       7/8      34867      401         53          25
shipfast  strands      4/8      26981      331         43          18
research  langgraph   10/10     44361     1464         57          18
research  crewai       9/10     60243     1194         62          22
research  strands      9/10     48803     1147         57          19


## Item by item

`P` passed, `F` failed; then total tokens and seconds for that scenario or question.

In [3]:
def matrix(agent):
    key = "scenarios" if agent == "shipfast" else "questions"
    ids = [i["id"] for i in results[(agent, "langgraph")][key]]
    print(f"{'':36}" + "".join(f"{fw:>22}" for fw in FRAMEWORKS))
    for iid in ids:
        row = f"{iid:36}"
        for fw in FRAMEWORKS:
            it = next(x for x in results[(agent, fw)][key] if x["id"] == iid)
            row += f"{('P' if it['passed'] else 'F'):>6} {it['input_tokens'] + it['output_tokens']:7} {it['latency_s']:6.0f}s"
        print(row)


matrix("shipfast")
print()
matrix("research")

                                                 langgraph                crewai               strands
S1_status_with_followup                  P    3042     58s     P    4663     70s     P    3238     56s
S2_lost_refund_auto                      P    3425     40s     P    4856     51s     P    3577     34s
S3a_late_refund_approved                 P    5234     55s     P    5406     59s     F    4142     57s
S3b_late_refund_rejected                 P    6906     74s     P    6895     79s     F    4142     52s
S4_damaged_needs_photo_then_refund       P    6360     93s     F    8587     82s     P    6619     78s
S5_outside_refund_window                 F    3341     44s     P    2720     38s     F    3509     35s
S6_unknown_order                         P    1490     19s     P    1576     19s     F    1590     17s
S7_out_of_scope                          P     164      4s     P     164      3s     P     164      3s

                                                 langgraph              

## Why each failure failed

In [4]:
for (agent, fw), data in results.items():
    for it in data["scenarios" if agent == "shipfast" else "questions"]:
        if not it["passed"]:
            print(f"{fw:10} {it['id']:36} {it['problems'][:2]}")

langgraph  S5_outside_refund_window             ["action: expected 'declined', got 'refunded'", 'refund_amount: expected None, got 35.0']
crewai     S4_damaged_needs_photo_then_refund   ["action: expected 'refunded', got 'answered'", 'refund_amount: expected 60.0, got None']
strands    S3a_late_refund_approved             ["action: expected 'refunded', got 'answered'", 'refund_amount: expected 250.0, got None']
strands    S3b_late_refund_rejected             ["action: expected 'escalated', got 'answered'"]
strands    S5_outside_refund_window             ["action: expected 'declined', got 'refunded'", 'refund_amount: expected None, got 35.0']
strands    S6_unknown_order                     ["action: expected 'needs_info', got 'declined'"]
crewai     Q10_order_cost                       ["missing facts: ['price_40', 'price_mini']"]
strands    Q07_deployment                       ["repeats stale or wrong figures: ['9 provinces']"]


## Agent code per build

Lines of code, not counting blank lines, comments or docstrings. "Shared plumbing" is each build's model
setup, tool wrappers, prompt re-exports, metrics hookup (and, for CrewAI, local memory and the defaults it
switches off). The contract's shared code (`derive_resolution`, the runners) is the same for all three and
not counted per build.

In [5]:
import ast
import io
import tokenize


def code_lines(path: Path) -> int:
    src = path.read_text()
    docstrings = set()
    for node in ast.walk(ast.parse(src)):
        body = getattr(node, "body", None)
        if isinstance(node, (ast.Module, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and body \
                and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0], "value", None), ast.Constant) \
                and isinstance(body[0].value.value, str):
            docstrings.update(range(body[0].lineno, body[0].end_lineno + 1))
    lines = set()
    skip = (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.INDENT, tokenize.DEDENT, tokenize.ENDMARKER)
    for tok in tokenize.generate_tokens(io.StringIO(src).readline):
        if tok.type not in skip:
            lines.update(range(tok.start[0], tok.end[0] + 1))
    return len(lines - docstrings)


PACKAGES = {"langgraph": "langgraph-tutorial/lg_agents", "crewai": "crewai-tutorial/crew_agents",
            "strands": "strands-tutorial/strands_agents_impl"}
GROUPS = {"ShipFast agent": ["shipfast.py", "approval.py"], "research agent": ["research_flow.py", "structured.py"],
          "shared plumbing": ["llm.py", "tools.py", "prompts.py", "metrics.py", "memory.py", "__init__.py"]}
print(f"{'':16}" + "".join(f"{fw:>11}" for fw in FRAMEWORKS))
for group, files in GROUPS.items():
    row = f"{group:16}"
    for fw in FRAMEWORKS:
        pkg = ROOT / PACKAGES[fw]
        row += f"{sum(code_lines(pkg / f) for f in files if (pkg / f).exists()):11}"
    print(row)

                  langgraph     crewai    strands
ShipFast agent          109        206         82
research agent           78         77         72
shared plumbing          34         79         43


## The experiment behind the Strands ShipFast gap

Strands failed both approval scenarios because its agent told the customer a refund was due and ended
its turn without calling `issue_refund`, so the approval gate never paused. A recording relay captured the
exact request each framework sent for the deciding model call in S3a. Everything matched (system prompt,
options, messages, tool calls, tool results) except one thing: Strands adds a placeholder description,
`"Parameter order_id"`, to every tool parameter that has none. The replay sends both requests, then each
with that one difference flipped.

In [6]:
import subprocess
import urllib.request

try:
    urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=3)
    print(subprocess.run(["python3", "-I", str(ROOT / "comparison/evidence/replay_tool_schema.py")],
                         capture_output=True, text=True, timeout=900).stdout)
except OSError:
    print("Ollama is not running; see COMPARISON.md for the recorded result.")

Strands request as sent               -> text: The order A1003 is delayed, and the ETA is October 24, 2026. Since tod
Strands request, descriptions removed -> calls ['issue_refund']
LangGraph request as sent             -> calls ['issue_refund']
LangGraph request, descriptions added -> text: The order A1003 is delayed, and the ETA is October 24, 2026. Since tod



The placeholder descriptions alone flip the model's decision, in both directions. The finding is
about small models and tool schemas, not about Strands' agent loop: Strands sends the same conversation,
and a content-free description was enough to push an 8B model from acting to talking.